# Word Embeddings

In this demonstration, we will see how to use pre-trained word embeddings from[`spacy`](https://spacy.io/) and [`gensim`](https://radimrehurek.com/gensim/).

We'll then see how to train our own embeddings.

In [ ]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy
import warnings
warnings.filterwarnings("ignore")

## Spacy pre-trained model

We are going to work with a small model for English, that contains embeddings for 20,000 words.

In [ ]:
!python -m spacy download en_core_web_sm

## Usage

In [ ]:
nlp = spacy.load('en_core_web_sm')

doc = nlp("This is some text that I am processing with Spacy")

print(doc[3].vector)

print("Average vector")
print(doc.vector)

## Gensim pre-trained model

Gensim does not provide pre-trained embeddings, but provides a loading mechanism for pre-trained embeddings instead.

Some sources for pre-trained embeddings:

- [Google News word2vec documentation](https://code.google.com/archive/p/word2vec/); [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)

- [NLPL word embeddings repository](http://vectors.nlpl.eu/repository/)

- [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- [French embeddings by Jean-Philippe Fauconnier](https://fauconnier.github.io/)

In [ ]:
!wget https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

In [ ]:
!ls -l 
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

## [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html) usage

In [ ]:
model = gensim.models.KeyedVectors.load_word2vec_format(model_file, binary=True)
print("Vocabulary size: ",len(model.vocab))

In [ ]:
for i, word in enumerate(model.vocab):
  print(word)
  if i > 50:
    break

In [ ]:
# Retrieve the vector for a word
vector = model['facile']

# Preprocessing needs to be the same as during training
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(' ')
vectors = [(word, model[word]) for word in words if word in model.vocab]
print("Transformed words:")
for word, vector in vectors:
  print(word)

In [ ]:
model.most_similar('berlin')

In [ ]:
print(model.similarity('berlin', 'munich'))
print(model.similarity('berlin', 'paris'))

## Operations in semantic space

Using [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector), we will define an analogy function:

`x` is to `y` what `z` is to?

*E.g.*: *King* is to *Queen* what *Uncle* is to?

Réponse : *Aunt*

In [ ]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

In [ ]:
semantic_analogy("homme", "femme", "chien")

## Word Embeddings training

### 20 Newsgroups dataset

20,000 “posts” split into 20 themes.

In [ ]:
!git clone https://github.com/nzmonzmp/20Newsgroups.git
!tar xzf 20Newsgroups/20news-bydate.tar.gz
!ls 20news-bydate-train/

In [ ]:
!ls 20news-bydate-test/

In [ ]:
contents = [p.read_text(encoding="latin-1")
            for p in pathlib.Path(".").glob("20news-bydate-*/*/*")]


print(f"{len(contents)} texts retrieved")

In [ ]:
print(contents[1])

In [ ]:
# Cleaning
def preprocess(content: str) -> typing.List[str]:
  offset = content.find('\n\n')
  if i > 0:
    content = content[offset + 2:]
  sentences = (re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
               for line in content.splitlines()
               if not line.endswith("writes:"))
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [ ]:
print(texts[0])

## Phrase detection using Gensim Phraser

In [ ]:
common_terms = ["of", "with", "without", "and", "or", "the", "a"]

phrases = gensim.models.phrases.Phrases(texts,
                                        common_terms=common_terms)

phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

In [ ]:
model = gensim.models.Word2Vec(
    phrased_texts, 
    min_count=3,
    size=200,
    workers=2,
    window=5,
    iter=30)

In [ ]:
model.most_similar('New_York')

In [ ]:
semantic_analogy("wind", "fly", "water")